# Unit 9 solutions: feature towers and item cold-start

These reference solutions build the feature inputs for a two-tower whose **item** side is rebuilt from *what a book is*, then train, register and honestly grade `FeatureTowerRetrievalPath`. Unit 8's two-tower gave every book a bare per-id `torch.nn.Embedding` row, and that row stays negative-shaped for a **cold item** (zero train positives): a cold book is never a positive, and under Unit 8's full-catalog negative sampler it is only ever a *negative*, so its id row is pushed down and the book is buried. The feature tower keeps the reader-tower / item-tower dot-product model but composes the item tower as a per-id embedding **plus** a learned genre embedding, a learned author embedding, and a `Linear` projection of the Unit-7 GloVe keyword vector -- all shared across books and trained from the warm items, so a cold book inherits a sensible vector from its content.

Read the result **honestly**. Measured at the pinned config (seed 0, k=10, 40 epochs, `negative_pool="warm"`), the feature tower scores warm hit@10 ~0.298 -- a little *below* the ID-only two-tower's ~0.340 (within 0.06), **not** a new warm record. The win is cold-item **coverage**: it lifts cold reach from 0 to ~0.138 (113 of the 818 zero-train books) while CF and the ID-only two-tower stay at exactly 0. It is not even the first path to serve cold items -- Unit 7's semantic content path already covers ~0.328 of them. The honest headline is narrower and true: the feature tower is the first **learned-taste / collaborative** path to reach cold items while holding near the book's best warm hit -- a warm/cold **compromise**, not dominance.

Everything is graded on the **validation** split; each reader's **seen** set is their train-positive history. The cold-coverage denominator and reader cohort are built from TRAIN positives alone (no held-out dependency), and the sealed `test` holdout is never touched here. These solutions run top-to-bottom with fixed seeds and keep their executed outputs. Training recurs, so a full 40-epoch fit (~20-27 s) is done **once** for the warm-only tower and *reused*; only the two Challenge exercises add a full fit each, and the determinism cell uses a deliberately small epoch count.

In [1]:
import json
from collections import defaultdict

import bookrec
import numpy as np
import pandas as pd
from bookrec import (
    FeatureTowerRetrievalPath,
    ItemItemRetrievalPath,
    PathRegistry,
    SemanticEmbeddingRetrievalPath,
    TwoTowerRetrievalPath,
    book_embedding,
    run_validation_scoreboard,
)
from bookrec.lexical import tokenize   # the library tokenizer (what book_embedding uses internally)

SEED = 0
K = 10        # every scoreboard reports at the top-10 cutoff

# --- the real interaction log + content (the same substrate as Units 1-8) ------------------
data_dir = bookrec.generated_dir()
catalog = bookrec.load_catalog(data_dir / "catalog.csv.gz")        # {item_id: Book} (2000 books)
catalog_ids = sorted(catalog)
interactions_path = data_dir / "interactions.csv.gz"
keywords = bookrec.load_keywords(data_dir / "keywords.csv.gz")     # {item_id: "space joined token bag"}
glove = bookrec.load_glove_subset()                                # the committed GloVe subset

cold = json.loads((data_dir / "cold_partitions.json").read_text())
cold_readers = set(cold["cold_readers"])      # 60 readers the personalized paths cannot serve

# The interaction log as plain row dicts -- the row-mapping log the paths duck-type.
interactions = pd.read_csv(interactions_path)
rows = interactions.to_dict("records")

# Each reader's train-positive history is their `seen` (exclusion) set; their val positives are the
# relevance target. We grade on `val`; no cell here reads the sealed `test` split.
seen_by = defaultdict(set)
val_pos_by = defaultdict(set)
train_pos_items = set()
for row in rows:
    if row["label"] != 1:
        continue
    if row["split"] == "train":
        seen_by[row["reader_id"]].add(row["item_id"])
        train_pos_items.add(row["item_id"])
    elif row["split"] == "val":
        val_pos_by[row["reader_id"]].add(row["item_id"])

# The 818 zero-train books -- books with NO train positive. This is the cold-coverage denominator,
# and it is built from TRAIN positives alone (no val / test dependency).
zero_train = set(catalog_ids) - train_pos_items

print(f"catalog books: {len(catalog_ids)}   zero-train (cold) books: {len(zero_train)}")
assert len(zero_train) == 818, "the validation-free cold denominator is the 818 zero-train books"

catalog books: 2000   zero-train (cold) books: 818


## Exercise 1 - A cold book has zero train positives, but it still has features

The whole idea of the feature tower is that a book is more than its id. Unit 8 gave every book a bare per-id embedding, which carries no useful signal for a book nobody has read -- its id row is only ever a sampled negative. The feature tower instead feeds the item tower *content*: the book's **genre(s)** (a multi-hot over the genre vocabulary), its **author**, and its **GloVe keyword vector** (the Unit-7 mean-pooled embedding). None of these needs a single interaction -- they come straight from the catalog -- so a cold book gets exactly the same kind of feature description as a popular one.

Below I build those feature inputs by hand for two books: a warm book and a **cold** book (the smallest id in `zero_train`). I build the catalog-wide genre vocabulary (sorted, exactly as the tower does), a `genre_multihot` for each book, read each book's author id, and compute each book's GloVe keyword vector with the shipped `book_embedding`. The punchline: the cold book, despite zero train positives, still carries genres and a nonzero GloVe vector.

In [2]:
# The catalog-wide genre vocabulary, sorted for determinism (exactly what the feature tower builds).
genre_vocab = sorted({
    token.strip()
    for item_id in catalog_ids
    for token in catalog[item_id].fields.get("genres", "").split(";")
    if token.strip()
})
genre_index = {genre: j for j, genre in enumerate(genre_vocab)}


def book_genres(item_id):
    """The book's genre tokens, parsed from the catalog's ';'-joined `genres` field."""
    raw = catalog[item_id].fields.get("genres", "")
    return [token.strip() for token in raw.split(";") if token.strip()]


def genre_multihot(item_id):
    """A multi-hot vector over genre_vocab: 1.0 in each slot the book's genres occupy."""
    vec = np.zeros(len(genre_vocab), dtype=np.float32)
    for genre in book_genres(item_id):
        vec[genre_index[genre]] = 1.0
    return vec


warm_book = min(train_pos_items)   # a book WITH train interactions (smallest id with a train positive)
cold_book = min(zero_train)     # a zero-train (cold) book -- NO train positives (only ever a sampled negative)

warm_genres = book_genres(warm_book)                             # a list of genre tokens
cold_genres = book_genres(cold_book)                             # the cold book still HAS genres
warm_author = catalog[warm_book].fields.get("author_id", "")
cold_author = catalog[cold_book].fields.get("author_id", "")
warm_glove = book_embedding(tokenize(keywords[warm_book]), glove)  # GloVe keyword vector
cold_glove = book_embedding(tokenize(keywords[cold_book]), glove)

warm_multihot = genre_multihot(warm_book)
cold_multihot = genre_multihot(cold_book)

# The punchline: the cold book has zero train positives, yet it still carries content features.
cold_in_zero_train = bool(cold_book in zero_train)
cold_has_features = bool(len(cold_genres) > 0 and bool(np.any(cold_glove != 0.0)))

feature_explanation = (
    "The genre embedding, author embedding and GloVe projection are SHARED parameters, learned once "
    "across all the warm books, so a cold book reuses those same trained weights to turn its own "
    "genres / author / keywords into a sensible vector. A bare per-id embedding has nothing of the "
    "kind: the cold book's id row receives only negative gradients (sampled as a negative, never a positive), so it stays "
    "negative-shaped and carries no useful taste signal. Content features describe what the book IS; an id "
    "row can only remember who has read it."
)

print(f"warm book {warm_book}: genres={warm_genres} author={warm_author!r} "
      f"multihot sum={int(warm_multihot.sum())}")
print(f"cold book {cold_book}: genres={cold_genres} author={cold_author!r} "
      f"multihot sum={int(cold_multihot.sum())}")
print(f"cold GloVe vector nonzero? {bool(np.any(cold_glove != 0.0))}  "
      f"(norm {float(np.linalg.norm(cold_glove)):.3f})")
print(f"cold book in zero_train? {cold_in_zero_train}   cold book has features? {cold_has_features}")

assert cold_in_zero_train, "the cold book is a genuine zero-train item"
assert cold_has_features, "the cold book still carries genres AND a nonzero GloVe keyword vector"
assert warm_multihot.shape == cold_multihot.shape == (len(genre_vocab),), (
    "both multi-hots live over the same sorted genre vocabulary"
)
assert warm_book in train_pos_items and warm_book != cold_book, (
    "the warm book genuinely has train interactions and differs from the cold book"
)


warm book 1: genres=['science', 'horror', 'adventure'] author='125' multihot sum=3
cold book 0: genres=['biography', 'poetry', 'adventure'] author='175' multihot sum=3
cold GloVe vector nonzero? True  (norm 1.000)
cold book in zero_train? True   cold book has features? True


## Exercise 2 - Train and register the feature tower (warm-only negatives)

Now the real path. `FeatureTowerRetrievalPath` trains a reader id-embedding tower and the composed item tower (id + genre + author + GloVe projection) by BPR + Adam + weight decay, exactly like Unit 8's two-tower but with the richer item side. Features arrive through the **constructor** (the catalog `Book` records, the keyword corpus and the GloVe subset), so `fit(rows, catalog=catalog_ids)` keeps the exact protocol signature -- `catalog` is an iterable of item ids, never `Book`s. The pinned defaults are the configuration measured on this data: `embedding_dim=32, n_epochs=40, learning_rate=0.01, n_negatives=10, batch_size=256, weight_decay=1e-4, negative_pool="warm", seed=0`.

The one design choice that matters most is `negative_pool`. The shipped default `"warm"` draws negatives **only from the train-positive (warm) item universe**, so a cold book is never sampled as a negative and its features are free to place it. (Exercise 7 shows what `"full"` does instead.) I fit the path **once** at the default config and reuse the fitted object through Exercises 3-6.

In [3]:
# One full fit at the pinned warm-only config (~20-27 s). REUSED by Exercises 3, 4, 5, 6 -- no refit.
# torch is imported lazily INSIDE fit; retrieve / load / artifact stay torch-free.
feature_tower = FeatureTowerRetrievalPath(
    catalog, keywords, glove, negative_pool="warm", seed=SEED
).fit(rows, catalog=catalog_ids)

losses = feature_tower.loss_history       # mean BPR loss per epoch (a list)
n_epochs_recorded = len(losses)           # 40, the pinned default
loss_decreased = bool(float(losses[-1]) < float(losses[0]))

# Register it as a first-class recommender (it must own the name 'feature-tower-v1').
registry = PathRegistry()
registry.register(feature_tower)
artifact_name = feature_tower.artifact_name()                 # 'feature-tower-v1'
registered = bool(artifact_name == "feature-tower-v1" and "feature-tower" in registry)

# A reader WITH a learned embedding gets K recs; `seen` only EXCLUDES already-read books.
warm_reader = feature_tower.artifact()["reader_ids"][0]
warm_recs = [c.item_id for c in feature_tower.retrieve(warm_reader, {"seen": seen_by[warm_reader]}, K)]
got_k_recs = bool(len(warm_recs) == K)

tower_explanation = (
    "The item tower now SUMS four signals into the shared space: the Unit-8 per-id embedding, a "
    "learned genre embedding (mean-pooled over the book's genres), a learned author embedding, and a "
    "Linear projection of the book's GloVe keyword vector. The genre / author / GloVe parameters are "
    "shared across every book and trained from the warm items, so a cold book -- whose own id row "
    "only ever receives negative gradients -- still receives a placed vector from its content features, which is exactly what "
    "lets it score at all."
)

print(f"epochs recorded: {n_epochs_recorded}")
print(f"BPR loss {float(losses[0]):.4f} -> {float(losses[-1]):.4f}  (decreased? {loss_decreased})")
print(f"registered as {artifact_name!r}? {registered}")
print(f"warm reader {warm_reader} got {len(warm_recs)} recs")

assert n_epochs_recorded == 40, "the pinned default trains 40 epochs"
assert loss_decreased, "the BPR loss must fall from the first epoch to the last"
assert registered, "the path must own and register as 'feature-tower-v1'"
assert got_k_recs, "a reader with a learned embedding gets K recommendations"

epochs recorded: 40
BPR loss 0.2188 -> 0.0688  (decreased? True)
registered as 'feature-tower-v1'? True
warm reader 0 got 10 recs


## Exercise 3 - The validation scoreboard: a small, honest warm cost

Score the feature tower beside the paths it extends: Unit 8's ID-only two-tower, Unit 4's item-item CF, and Unit 7's semantic content path, all at hit@10 on `val` (the `feature_tower` is reused from Exercise 2, not refit). I keep the three comparison paths fitted -- Exercise 4 reuses them.

Read the numbers **honestly, and the honest reading is NOT a warm win**. The feature tower lands around hit@10 ~0.298 -- a little *below* the ID-only two-tower's ~0.340 (within 0.06), above CF (~0.252) and far above semantic (~0.102). It does not set a warm record; it pays a few points of warm hit relative to the plain two-tower. That trade buys cold-item reach, which Exercise 4 measures.

In [4]:
# Fit the comparison paths. The two-tower is the only other slow fit (~13 s); CF and semantic are fast.
# The feature_tower itself is REUSED from Exercise 2 (no refit). KEEP these three for Exercise 4.
two_tower = TwoTowerRetrievalPath(seed=SEED).fit(rows, catalog=catalog_ids)       # Unit 8 ID-only
item_item = ItemItemRetrievalPath().fit(rows, catalog=catalog_ids)                # Unit 4 CF
semantic = SemanticEmbeddingRetrievalPath(keywords, glove).fit(rows, catalog=catalog_ids)  # Unit 7


def val_hit(path):
    return run_validation_scoreboard(
        path, interactions_path, catalog_ids=catalog_ids, k=K, cold_readers=cold_readers
    ).hit_rate_at_k


ft_hit = val_hit(feature_tower)   # ~0.298 (a small cost vs the two-tower)
tt_hit = val_hit(two_tower)       # ~0.340 (the warm leader)
cf_hit = val_hit(item_item)       # ~0.252
sem_hit = val_hit(semantic)       # ~0.102

feature_tower_not_best = bool(ft_hit < tt_hit)          # NOT a warm record
within_six_points = bool(ft_hit >= tt_hit - 0.06)       # but close -- not tanked
beats_cf = bool(ft_hit > cf_hit)

honest_warm_reading = (
    "The warm ordering is two-tower (~0.340) > feature tower (~0.298) > item-item CF (~0.252) > "
    "semantic (~0.102). The feature tower is within 0.06 of the best but does NOT beat it -- it is "
    "the plain two-tower minus a few points of warm hit, the price of rebuilding the item tower from "
    "content. A path accepts that small warm cost on purpose when it buys something warm hit cannot "
    "see: reach to the cold items that have no train-positive history, which Exercise 4 measures."
)

print(f"feature tower hit@{K} = {ft_hit:.3f}  (within 0.06 of best, but NOT the best)")
print(f"two-tower     hit@{K} = {tt_hit:.3f}  (the warm leader)")
print(f"item-item CF  hit@{K} = {cf_hit:.3f}")
print(f"semantic      hit@{K} = {sem_hit:.3f}")

assert ft_hit >= 0.28, "the feature tower warm hit stays near the book's best (>= 0.28)"
assert feature_tower_not_best, "honest: the feature tower does NOT beat the two-tower on warm"
assert within_six_points, "but it is within 0.06 of the best -- a small cost, not a tank"
assert beats_cf, "it clears item-item CF on warm hit"

feature tower hit@10 = 0.298  (within 0.06 of best, but NOT the best)
two-tower     hit@10 = 0.340  (the warm leader)
item-item CF  hit@10 = 0.252
semantic      hit@10 = 0.102


## Exercise 4 - Cold-item coverage: the metric that matters here

The designated cold items have no `val` positives by construction, so cold **relevance** cannot be graded on `val` (only the sealed `test` could, at Checkpoint B). The validation-safe metric is cold-item **coverage**: of the 818 zero-train books, how many appear in *some* fitted reader's top-10. The cohort and denominator are both built from TRAIN alone -- score every fitted non-cold reader from their train history and count the unique zero-train ids that land in a top-10, over all 818. This never depends on who happens to have a val positive.

The ID-only two-tower and CF come in at **exactly 0** (their cold items get negative-shaped / unseen embeddings and are buried), the feature tower at ~0.138 (113/818), and semantic **high** at ~0.328 -- content already serves cold items freely.

In [5]:
# Cohort = ALL fitted non-cold readers, scored from TRAIN history only (no val dependency).
def fitted_cohort(path):
    try:                                     # the tower paths expose fitted reader ids in the artifact
        fitted = [int(r) for r in path.artifact()["reader_ids"]]
    except KeyError:                         # item-item CF has no reader embedding -> use the train cohort
        fitted = sorted(seen_by)
    return [r for r in fitted if r not in cold_readers]


def cold_coverage(path):
    """(coverage fraction, count): unique zero-train ids in any top-10 / 818, train-only scoring."""
    surfaced = set()
    for reader_id in fitted_cohort(path):
        for c in path.retrieve(reader_id, {"seen": seen_by[reader_id]}, K):
            if c.item_id in zero_train:
                surfaced.add(c.item_id)
    return len(surfaced) / len(zero_train), len(surfaced)


denom = len(zero_train)                        # 818, the validation-free denominator
ft_cov, ft_n = cold_coverage(feature_tower)    # ~0.138 (113)
tt_cov, tt_n = cold_coverage(two_tower)        # exactly 0.0 (0)
cf_cov, cf_n = cold_coverage(item_item)        # exactly 0.0 (0)
sem_cov, sem_n = cold_coverage(semantic)       # ~0.328 (content already serves cold)

id_only_and_cf_bury_cold = bool(tt_cov == 0.0 and cf_cov == 0.0)
feature_tower_reaches_cold = bool(ft_cov > 0.0 and ft_cov > tt_cov)
semantic_already_serves_cold = bool(sem_cov > ft_cov)

coverage_reading = (
    "Cold coverage orders semantic (~0.328) > feature tower (~0.138) > two-tower = CF = 0. The ID-only "
    "two-tower and CF bury every one of the 818 cold books: with no train positives a cold item "
    "has only a negative-shaped id row (or zero co-occurrence), so it never reaches a top-10. The "
    "honest headline is NOT that the feature tower is the first to serve cold items -- Unit 7's "
    "content path already does, freely, and reaches more of them. It is the first LEARNED-taste / "
    "collaborative path to reach cold items while holding near the book's best warm hit: a "
    "warm/cold compromise, not dominance."
)

print(f"denominator (zero-train books): {denom}")
print(f"feature tower cold coverage = {ft_cov:.3f} ({ft_n}/{denom})")
print(f"two-tower     cold coverage = {tt_cov:.3f} ({tt_n}/{denom})")
print(f"item-item CF  cold coverage = {cf_cov:.3f} ({cf_n}/{denom})")
print(f"semantic      cold coverage = {sem_cov:.3f} ({sem_n}/{denom})")

assert denom == 818, "the validation-free cold denominator is 818"
assert id_only_and_cf_bury_cold and tt_n == 0 and cf_n == 0, "ID-only two-tower and CF reach zero cold items"
assert feature_tower_reaches_cold, "the feature tower surfaces cold items from their features (> 0, > ID-only)"
assert semantic_already_serves_cold, "content (semantic) already serves MORE cold items than the feature tower"

denominator (zero-train books): 818
feature tower cold coverage = 0.138 (113/818)
two-tower     cold coverage = 0.000 (0/818)
item-item CF  cold coverage = 0.000 (0/818)
semantic      cold coverage = 0.328 (268/818)


## Exercise 5 - One concrete cold book the feature tower surfaces (that the two-tower buries)

Coverage is a count; make it concrete for one reader. Among the readers who are not designated cold readers, some have a `val` positive that is a zero-train book they went on to read next -- there are exactly 97 such **incidental-cold-val** readers on this data. I build that cohort deterministically (sorted), then find the smallest-id reader in it for whom the feature tower surfaces, inside its top-10, one of that reader's own cold `val` positives. (This is the report-only cold hit; it is small, ~0.010, so only a few such readers exist.)

For that reader I show the surfaced cold book's id and title, and confirm the honest contrast: the **same** reader's ID-only two-tower top-10 and item-item CF top-10 contain **no** zero-train book at all.

In [6]:
# The 97 incidental-cold-val readers: non-cold readers with an UNSEEN zero-train val positive.
incidental_cohort = sorted(
    reader_id
    for reader_id in val_pos_by
    if reader_id not in cold_readers
    and ((val_pos_by[reader_id] - seen_by[reader_id]) & zero_train)
)
cohort_size = len(incidental_cohort)   # 97


def surfaced_cold_val_positive(path, reader_id):
    """A cold val-positive book this path puts in the reader's top-10, or None."""
    cold_relevant = (val_pos_by[reader_id] - seen_by[reader_id]) & zero_train
    top = {c.item_id for c in path.retrieve(reader_id, {"seen": seen_by[reader_id]}, K)}
    hit = top & cold_relevant
    return min(hit) if hit else None


# The deterministic reader: the smallest-id cohort reader whose feature tower surfaces a cold book
# they actually read next.
example_reader = next(
    r for r in incidental_cohort if surfaced_cold_val_positive(feature_tower, r) is not None
)
surfaced_book = surfaced_cold_val_positive(feature_tower, example_reader)
surfaced_title = catalog[surfaced_book].title
surfaced_is_cold = bool(surfaced_book in zero_train)


# The honest contrast on the SAME reader: the two-tower and CF surface NO zero-train book at all.
def cold_in_top(path, reader_id):
    return [
        c.item_id
        for c in path.retrieve(reader_id, {"seen": seen_by[reader_id]}, K)
        if c.item_id in zero_train
    ]


tt_cold_for_reader = cold_in_top(two_tower, example_reader)   # expect []
cf_cold_for_reader = cold_in_top(item_item, example_reader)   # expect []
only_feature_tower_surfaces_cold = bool(
    len(tt_cold_for_reader) == 0 and len(cf_cold_for_reader) == 0
)

cold_example_explanation = (
    "The feature tower scores the cold book from its shared genre / author / GloVe features -- "
    "parameters trained on the warm books -- so it can rank a never-read book into this reader's "
    "top-10 on content alone. The two-tower and CF saw that same book only as a sampled "
    "negative: the two-tower has only a negative-shaped id row (trained solely as a sampled negative) and CF has zero "
    "co-occurrence, so both bury it. Same reader, same cold book -- only the feature tower reaches it."
)

print(f"incidental-cold-val cohort size: {cohort_size}")
print(f"example reader {example_reader}: feature tower surfaced cold book {surfaced_book} "
      f"({surfaced_title!r})")
print(f"same reader -- two-tower cold books in top-10: {tt_cold_for_reader}")
print(f"same reader -- item-item CF cold books in top-10: {cf_cold_for_reader}")

assert cohort_size == 97, "the incidental-cold-val cohort is fixed by the data at 97 readers"
assert surfaced_is_cold, "the surfaced book is a genuine zero-train (cold) book"
assert only_feature_tower_surfaces_cold, "on the same reader, the two-tower and CF surface no cold book"

incidental-cold-val cohort size: 97
example reader 129: feature tower surfaced cold book 139 ('Book 00139')
same reader -- two-tower cold books in top-10: []
same reader -- item-item CF cold books in top-10: []


## Exercise 6 - Determinism and the retrieve contract

A trained model you cannot reproduce is not science, and a retrieval path has a contract to honor. `FeatureTowerRetrievalPath.fit` seeds `torch.manual_seed` and a numpy RNG, enables `torch.use_deterministic_algorithms(True)` and pins `torch.set_num_threads(1)` (saving and restoring the last two process-global switches), so two fits at the same seed agree. Determinism holds at any epoch count, so I fit **two** small towers here (a handful of epochs, a few seconds each) rather than two full 40-epoch fits.

I confirm the two small fits are `allclose` on both learned matrices and give an **identical** top-K ranking for a few readers (that tolerance-plus-ranking match is the gate; bit-identical floats are a CPU bonus, not the standard). Then I check the retrieve contract on the `feature_tower` from Exercise 2: a known reader with an empty `seen` still gets `K` recs, and an unknown reader gets `[]`.

In [7]:
# Determinism holds at any epoch count, so use a SMALL n_epochs here (two cheap fits, a few seconds
# each) instead of two full 40-epoch fits -- the reproducibility gate is identical.
SMALL_EPOCHS = 5
fit_a = FeatureTowerRetrievalPath(catalog, keywords, glove, n_epochs=SMALL_EPOCHS, seed=SEED).fit(rows, catalog=catalog_ids)
fit_b = FeatureTowerRetrievalPath(catalog, keywords, glove, n_epochs=SMALL_EPOCHS, seed=SEED).fit(rows, catalog=catalog_ids)

readers_allclose = bool(np.allclose(fit_a.reader_embeddings, fit_b.reader_embeddings))
items_allclose = bool(np.allclose(fit_a.item_embeddings, fit_b.item_embeddings))

# Identical top-K ranking for the first few readers (the reproducibility gate).
reader_ids = fit_a.artifact()["reader_ids"][:5]
rankings_identical = bool(all(
    [c.item_id for c in fit_a.retrieve(r, {"seen": set()}, K)]
    == [c.item_id for c in fit_b.retrieve(r, {"seen": set()}, K)]
    for r in reader_ids
))
# Bonus (holds on CPU here, but NOT the gate): the weights are even bit-identical.
bit_identical = bool(np.array_equal(fit_a.item_embeddings, fit_b.item_embeddings))

# The retrieve contract, on the fitted `feature_tower` from Exercise 2.
known_reader = feature_tower.artifact()["reader_ids"][0]
empty_seen_recs = feature_tower.retrieve(known_reader, {"seen": set()}, K)
known_gets_k = bool(len(empty_seen_recs) == K)
unknown_reader = next(r for r in range(10_000_000) if r not in set(feature_tower.artifact()["reader_ids"]))
unknown_gets_empty = bool(feature_tower.retrieve(unknown_reader, {"seen": set()}, K) == [])

determinism_explanation = (
    "We gate on identical top-K RANKING plus allclose matrices -- a tolerance-based standard -- rather "
    "than demanding exactly equal floats, because what a recommender serves is the ORDER of the "
    "results, and tiny float differences that never change that order are harmless. Insisting on "
    "bit-identical weights would be brittle (a different CPU, BLAS build or thread count can perturb "
    "the last bits) while buying nothing the ranking gate does not already guarantee; bit-identity "
    "holds here on a single-threaded CPU, so we keep it only as a bonus."
)

print(f"reader embeddings allclose? {readers_allclose}")
print(f"item embeddings allclose?   {items_allclose}")
print(f"top-{K} rankings identical for {len(reader_ids)} readers? {rankings_identical}")
print(f"bonus: weights bit-identical? {bit_identical}")
print(f"known reader empty-seen recs: {len(empty_seen_recs)} (== K? {known_gets_k})")
print(f"unknown reader {unknown_reader} returns []? {unknown_gets_empty}")

assert readers_allclose and items_allclose, "two seeded fits give allclose reader AND item embeddings"
assert rankings_identical, "the reproducibility gate: identical top-K ranking across fits"
assert known_gets_k, "a known reader with empty seen still gets K recs"
assert unknown_gets_empty, "an unknown / cold reader gets []"

reader embeddings allclose? True
item embeddings allclose?   True
top-10 rankings identical for 5 readers? True
bonus: weights bit-identical? True
known reader empty-seen recs: 10 (== K? True)
unknown reader 13 returns []? True


## Exercise 7 - Challenge: the negative pool is the whole mechanism

Here is the decisive design choice. 41% of this catalog is zero-train, so if negatives are sampled uniformly over the **full** catalog (Unit 8's sampler), a cold book is *only ever* a negative: the tower gets a push-down gradient on it every epoch and no positive to balance it, and features can only weakly counter a book's own negative gradient (full-pool coverage collapses to ~0.048, not to 0). Drawing negatives only from the **warm** (train-positive) universe removes that push-down, so a cold book's features are free to place it. The knob is `negative_pool`, and it is an honest **warm-cold trade**, not a free lunch.

I refit the feature tower once with `negative_pool="full"` (everything else pinned; ~20 s) and compare it to the warm-only `feature_tower` from Exercise 2 on both axes -- warm hit@10 and cold coverage. `"full"` *raises* warm hit a touch (~0.338, almost the plain two-tower) while *collapsing* cold coverage (~0.048, 39/818); `"warm"` gives up a little warm (~0.298) to roughly triple cold coverage (~0.138).

In [8]:
# One extra full fit (~20 s): the SAME tower with Unit 8's full-catalog negative sampler.
# The warm-only `feature_tower` from Exercise 2 is REUSED -- do not refit it.
full_pool_tower = FeatureTowerRetrievalPath(
    catalog, keywords, glove, negative_pool="full", seed=SEED
).fit(rows, catalog=catalog_ids)

# Reuse val_hit (Exercise 3) and cold_coverage (Exercise 4).
warm_pool_hit = val_hit(feature_tower)        # ~0.298
full_pool_hit = val_hit(full_pool_tower)      # ~0.338 (higher warm)
warm_pool_cov, warm_pool_n = cold_coverage(feature_tower)     # ~0.138 (113)
full_pool_cov, full_pool_n = cold_coverage(full_pool_tower)   # ~0.048 (39)  cold collapses

full_raises_warm = bool(full_pool_hit > warm_pool_hit)
warm_raises_cold = bool(warm_pool_cov > full_pool_cov)

negative_pool_explanation = (
    "Full-catalog negatives make the 818 cold books negatives-only: every epoch each cold book gets a "
    "push-down gradient and never a balancing positive, so however informative its features, "
    "the tower suppresses it hard -- cold coverage collapses from ~0.138 to ~0.048 (not to 0: features still surface 39 of 818). In exchange, "
    "full negatives edge warm hit up (~0.298 -> ~0.338, almost the plain two-tower), because the warm "
    "ranker sees a broader negative distribution. Warm-only negatives reverse the trade: a cold book "
    "is never a negative, so its features place it (cold ~0.138) at the cost of a few warm points. It "
    "is a genuine warm-cold trade, not a free lunch."
)

print(f"warm-only negatives: hit={warm_pool_hit:.3f}  cold cov={warm_pool_cov:.3f} ({warm_pool_n}/818)")
print(f"full-catalog negatives: hit={full_pool_hit:.3f}  cold cov={full_pool_cov:.3f} ({full_pool_n}/818)")

assert full_raises_warm, "full-catalog negatives raise warm hit a touch (toward the plain two-tower)"
assert warm_raises_cold, "warm-only negatives give far higher cold coverage (features can place cold books)"

warm-only negatives: hit=0.298  cold cov=0.138 (113/818)
full-catalog negatives: hit=0.338  cold cov=0.048 (39/818)


## Exercise 8 - Challenge: hard negatives trade even harder (and the false-negative caveat)

A tempting idea is to make the negatives *harder* -- instead of uniform-over-warm, sample popular warm books (which a reader is more likely to have plausibly considered) more often. `negative_pool="hard"` does exactly that (popularity-weighted over the warm universe). The folklore is that harder negatives *sharpen* a ranker; the honest measurement on this data is that they do **not** sharpen both axes -- they push the warm-cold trade even further toward cold, at a real warm cost. There is also a correctness caveat: a popular book a reader has not logged might actually be a true (unobserved) positive, so hard sampling risks training against a **false negative**.

I refit once with `negative_pool="hard"` (everything else pinned; ~24 s) and compare it to the uniform warm-only `feature_tower` on warm hit@10 and cold coverage. Hard negatives *lower* warm hit (~0.172, well below the uniform ~0.298) while *raising* cold coverage further (~0.226, 185/818).

In [9]:
# One extra full fit (~24 s): popularity-weighted ('hard') negatives over the same warm universe.
# The uniform warm-only `feature_tower` from Exercise 2 is REUSED -- do not refit it.
hard_pool_tower = FeatureTowerRetrievalPath(
    catalog, keywords, glove, negative_pool="hard", seed=SEED
).fit(rows, catalog=catalog_ids)

uniform_hit = val_hit(feature_tower)        # ~0.298 (uniform warm-only)
hard_hit = val_hit(hard_pool_tower)         # ~0.172 (lower warm)
uniform_cov, uniform_n = cold_coverage(feature_tower)      # ~0.138 (113)
hard_cov, hard_n = cold_coverage(hard_pool_tower)          # ~0.226 (185)  cold rises further

hard_lowers_warm = bool(hard_hit < uniform_hit)
hard_raises_cold = bool(hard_cov > uniform_cov)
hard_did_not_sharpen = bool(hard_lowers_warm and hard_raises_cold)   # traded, not sharpened

hard_negative_explanation = (
    "Hard (popularity-weighted) negatives traded warm hit DOWN (~0.298 -> ~0.172) for more cold reach "
    "(~0.138 -> ~0.226): they did NOT sharpen both axes, they pushed the warm-cold trade even further "
    "toward cold. Harder negatives concentrate the push-down gradient on popular warm books, which "
    "sharpens the boundary against those but costs ordinary warm accuracy. And a popular book a reader "
    "has not logged may in fact be a true (unobserved) positive, so hard sampling can train the model "
    "AGAINST books the reader would actually like -- a false negative -- which is why aggressive hard "
    "sampling is something to use with care."
)

print(f"uniform warm-only: hit={uniform_hit:.3f}  cold cov={uniform_cov:.3f} ({uniform_n}/818)")
print(f"hard (popularity): hit={hard_hit:.3f}  cold cov={hard_cov:.3f} ({hard_n}/818)")

assert hard_lowers_warm, "hard negatives lower warm hit (a real warm cost)"
assert hard_raises_cold, "hard negatives raise cold coverage further than uniform warm-only"
assert hard_did_not_sharpen, "hard negatives TRADED warm for cold -- they did not sharpen both axes"

uniform warm-only: hit=0.298  cold cov=0.138 (113/818)
hard (popularity): hit=0.172  cold cov=0.226 (185/818)
